# 06 - Exploratory, descriptive and sensitivity analyses

1. **Exploratory:** within AD, does the EEG score track global cognitive severity? Partial Spearman correlation
   (MoCA and, secondarily, IFS in BrainLat, adjusted for age and education; MMSE in the Greek cohort, adjusted for age).
2. **Descriptive:** which of the 11 features differ beyond the demographics, and in the same direction in both cohorts?
   (Each feature's subject mean, standardised; OR per 1 SD; Benjamini–Hochberg q within cohort.)
3. **Sensitivity:** only the 7 spectral features; without gamma; without epochs with gamma relative power > 0.15;
   SVM and XGBoost. Each variant re-runs the LOSO and the transport; the column "changes the reading of" compares its
   analytic p-values and transport CIs with those of the main analysis.

**Outputs:** S4, S5, S6 tables; outputs/results/sensitivity.json.

**Time:** 30-60 min the first time (cached afterwards).

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

from eegad.config import load_context
from eegad.data import load_table
from eegad.reporting import fmt, fmt_ci, fmt_metrics, fmt_p, read_json, write_json

import re

from scipy import stats as st

from eegad.models import model_factories, run_loso, transport
from eegad.stats import (auc_ci, benjamini_hochberg, incremental_test, nested_lr, partial_spearman_ci, safe_logit,
                         standardize)

ctx = load_context("06")
log, out, plan = ctx.log, ctx.out, ctx.plan
NB, SEED = ctx.n_boot, ctx.seed
FEATURES, COV_BL, COV_GR = ctx.features, ctx.cov_brainlat, ctx.cov_greek
df_bl, df_gr = load_table(out / "derived" / "brainlat_epochs.csv"), load_table(out / "derived" / "greek_epochs.csv")


def with_score(subjects, name):
    pred = load_table(out / "predictions" / f"oof_{name}_RF_11.csv")
    return subjects.merge(pred[["subject_id", "p"]].rename(columns={"p": "p_eeg"}), on="subject_id", how="left")


subj_bl = with_score(load_table(out / "derived" / "brainlat_subjects.csv"), "BrainLat")
subj_gr = with_score(load_table(out / "derived" / "greek_subjects.csv"), "Greek")

## 1) Exploratory: EEG score and cognitive severity within AD (Suppl. Table S4)

In [ ]:
EXPL = []
if plan["exploratory"]["run"]:
    for cohort, key, s, col, covs, label in (("BrainLat", "brainlat", subj_bl, "moca_total", ["Age", "years_education"], "MoCA"),
                                             ("BrainLat", "brainlat", subj_bl, "ifs_total_score", ["Age", "years_education"],
                                              "IFS (secondary)"),
                                             ("Greek", "greek", subj_gr, "mmse", ["Age"], "MMSE")):
        ad = s[s.y == 1].dropna(subset=[col, "p_eeg"] + covs)
        if len(ad) < 8:
            print(f"{cohort}: only {len(ad)} AD with {label}; skipped.")
            continue
        x = safe_logit(ad["p_eeg"])
        rho0 = st.spearmanr(x, ad[col])[0]
        rho, lo, hi, p = partial_spearman_ci(x, ad[col].to_numpy(float), ad[covs].to_numpy(float), NB, SEED)
        EXPL.append({"cohort": cohort, "scale": label, "n": len(ad), "rho_simple": rho0, "rho_partial": rho,
                     "lo": lo, "hi": hi, "p": p, "covariates": " + ".join(covs)})
        print(f"{cohort}: AD with {label} (n = {len(ad)}): simple rho {rho0:.3f}; partial rho ({' + '.join(covs)}) "
              f"{fmt_ci(rho, lo, hi)}, p = {fmt_p(p)}")
        log.record("Exploratory", f"{cohort}: partial rho between EEG score and {label} in AD",
                   f"{fmt_ci(rho, lo, hi)}, p = {fmt_p(p)}; n = {len(ad)}", "Results (exploratory)")
        log.keys_from(f"exploratory.{key}.{label.split(' ')[0].lower()}",
                      {"n": len(ad), "rho_simple": rho0, "rho_partial": rho, "lo": lo, "hi": hi, "p": p})
    if EXPL:
        pd.DataFrame(EXPL).to_csv(out / "tables" / "S4_exploratory_severity.csv", index=False, encoding="utf-8-sig")
else:
    print("Disabled in the analysis plan.")

## 2) Descriptive: which features differ beyond the demographics? (Suppl. Table S6)

In [ ]:
FEAT_ROWS = []
for cohort, d, s, covs in (("BrainLat", df_bl, subj_bl, COV_BL), ("Greek", df_gr, subj_gr, COV_GR)):
    means = d.groupby("subject_id")[FEATURES].mean().reset_index()
    t = s[["subject_id", "y"] + covs].merge(means, on="subject_id").dropna().reset_index(drop=True)
    yv = t["y"].to_numpy(int)
    Xr = np.column_stack([np.ones(len(t)), standardize(t[covs].to_numpy(float))])
    for f in FEATURES:
        Xf = np.column_stack([Xr, standardize(t[[f]].to_numpy(float))[:, 0]])
        r = nested_lr(Xr, Xf, yv, Xf.shape[1] - 1)
        FEAT_ROWS.append({"cohort": cohort, "feature": f, "n": len(t), "OR": r["OR"], "lo": r["OR_lo"],
                          "hi": r["OR_hi"], "p": r["p"], "method": r["method"]})
TAB_FEAT = pd.DataFrame(FEAT_ROWS)
TAB_FEAT["q_BH"] = TAB_FEAT.groupby("cohort")["p"].transform(benjamini_hochberg)
wide = TAB_FEAT.pivot(index="feature", columns="cohort").reindex(FEATURES)
S6 = pd.DataFrame({
    "BrainLat OR/SD [95% CI]": [fmt_ci(*wide.loc[f, [("OR", "BrainLat"), ("lo", "BrainLat"), ("hi", "BrainLat")]], 2)
                                for f in FEATURES],
    "BrainLat q": [fmt_p(wide.loc[f, ("q_BH", "BrainLat")]) for f in FEATURES],
    "Greek OR/SD [95% CI]": [fmt_ci(*wide.loc[f, [("OR", "Greek"), ("lo", "Greek"), ("hi", "Greek")]], 2)
                             for f in FEATURES],
    "Greek q": [fmt_p(wide.loc[f, ("q_BH", "Greek")]) for f in FEATURES],
    "same direction": [bool((wide.loc[f, ("OR", "BrainLat")] - 1) * (wide.loc[f, ("OR", "Greek")] - 1) > 0)
                       for f in FEATURES]}, index=FEATURES)
display(S6)
S6.to_csv(out / "tables" / "S6_features_beyond_demographics.csv", encoding="utf-8-sig")
both = [f for f in FEATURES if wide.loc[f, ("q_BH", "BrainLat")] < 0.05 and wide.loc[f, ("q_BH", "Greek")] < 0.05
        and S6.loc[f, "same direction"]]
print(f"Same direction in both cohorts: {int(S6['same direction'].sum())} of {len(FEATURES)} features | "
      f"q < 0.05 in both and in the same direction: {both or 'none'}")
log.record("Descriptive", "Features beyond the demographics (OR per SD; BH within cohort)",
           f"same direction in {int(S6['same direction'].sum())} of {len(FEATURES)}; q < 0.05 in both: "
           f"{', '.join(both) or 'none'}", "Discussion; Suppl. Table S6", "descriptive")
for _, r in TAB_FEAT.iterrows():
    log.keys_from(f"features.{r['feature']}.{r['cohort'].lower()}",
                  {"or.value": r["OR"], "or.lo": r["lo"], "or.hi": r["hi"], "q": r["q_BH"]})

## 3) Sensitivity analyses (Suppl. Table S5)

In [ ]:
inc = read_json(out / "results" / "incremental.json")["INC"]
main_bl, main_gr = load_table(out / "predictions" / "oof_BrainLat_RF_11.csv"), load_table(out / "predictions" / "oof_Greek_RF_11.csv")
tr_a, tr_b = load_table(out / "predictions" / "transport_RQ3a.csv"), load_table(out / "predictions" / "transport_RQ3b.csv")
REFERENCE = {"RQ1a": "adds" if inc["BrainLat"]["p"] < 0.05 else "not demonstrated",          # analytic p of the main
             "RQ1b": "adds" if inc["Greek"]["p_one_sided"] < 0.05 else "not demonstrated",
             "RQ3a": log.get_verdict("RQ3a"), "RQ3b": log.get_verdict("RQ3b")}
VARIANT_KEY = {"Main (RF, 11 features)": "main", "RF, 7 spectral features": "rf_spectral7",
               "RF, 10 features (no gamma)": "rf_no_gamma", "SVM, 11 features": "svm", "XGBoost, 11 features": "xgboost"}
subj_bl_base = load_table(out / "derived" / "brainlat_subjects.csv")
subj_gr_base = load_table(out / "derived" / "greek_subjects.csv")


def variant_summary(name, model, feats, gamma=None):
    dbl = df_bl if gamma is None else df_bl[df_bl["Rel_Gamma_mean"] <= gamma].reset_index(drop=True)
    dgr = df_gr if gamma is None else df_gr[df_gr["Rel_Gamma_mean"] <= gamma].reset_index(drop=True)
    sfx = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    if name == "Main (RF, 11 features)":
        pbl, pgr, tbg, tgb = main_bl, main_gr, tr_a, tr_b
    else:
        pbl = run_loso(dbl, feats, model, f"BL_{sfx}", ctx)
        pgr = run_loso(dgr, feats, model, f"GR_{sfx}", ctx)
        tbg = transport(dbl, dgr, feats, model, f"S_{sfx}_BL_to_GR", ctx).merge(subj_gr_base[["subject_id", "y"]], on="subject_id")
        tgb = transport(dgr, dbl, feats, model, f"S_{sfx}_GR_to_BL", ctx).merge(subj_bl_base[["subject_id", "y"]], on="subject_id")
    sb = subj_bl_base[["subject_id"] + COV_BL].merge(pbl[["subject_id", "y", "p"]], on="subject_id")
    sg = subj_gr_base[["subject_id"] + COV_GR].merge(pgr[["subject_id", "y", "p"]], on="subject_id")
    ib, ig = incremental_test(sb, COV_BL, "p", "BrainLat"), incremental_test(sg, COV_GR, "p", "Greek")
    a_bl, a_gr = auc_ci(sb["y"], sb["p"], NB, SEED), auc_ci(sg["y"], sg["p"], NB, SEED)
    a_bg, a_gb = auc_ci(tbg["y"], tbg["p"], NB, SEED), auc_ci(tgb["y"], tgb["p"], NB, SEED)
    v = {"RQ1a": "adds" if ib["p"] < 0.05 else "not demonstrated",
         "RQ1b": "adds" if ig["p_one_sided"] < 0.05 else "not demonstrated",
         "RQ3a": "transports" if a_bg[1] > 0.5 else "not demonstrated",
         "RQ3b": "transports" if a_gb[1] > 0.5 else "not demonstrated"}
    changes = [k for k in v if v[k] != REFERENCE[k]]
    key = VARIANT_KEY.get(name, "rf_gamma_epochs_removed")
    for label, (a, lo, hi) in (("auc_brainlat", a_bl), ("auc_greek", a_gr), ("rq3a_auc", a_bg), ("rq3b_auc", a_gb)):
        log.keys_from(f"sensitivity.{key}.{label}", {"value": a, "lo": lo, "hi": hi})
    log.keys_from(f"sensitivity.{key}", {"rq1a_or.value": ib["OR"], "rq1a_or.lo": ib["OR_lo"], "rq1a_or.hi": ib["OR_hi"],
                                         "rq1a_p": ib["p"], "rq1b_or.value": ig["OR"], "rq1b_or.lo": ig["OR_lo"],
                                         "rq1b_or.hi": ig["OR_hi"], "rq1b_p_one_sided": ig["p_one_sided"]})
    return {"variant": name, "subjects BL/GR": f"{len(sb)}/{len(sg)}",
            "AUC BrainLat": fmt_ci(*a_bl), "RQ1a OR/SD": fmt_ci(ib["OR"], ib["OR_lo"], ib["OR_hi"], 2),
            "RQ1a p": fmt_p(ib["p"]), "AUC Greek": fmt_ci(*a_gr),
            "RQ1b OR/SD": fmt_ci(ig["OR"], ig["OR_lo"], ig["OR_hi"], 2), "RQ1b one-sided p": fmt_p(ig["p_one_sided"]),
            "RQ3a BL->GR": fmt_ci(*a_bg), "RQ3b GR->BL": fmt_ci(*a_gb),
            "changes the reading of": ", ".join(changes) or "none",
            "_or": {"BrainLat": (ib["OR"], ib["OR_lo"], ib["OR_hi"]), "Greek": (ig["OR"], ig["OR_lo"], ig["OR_hi"])}}


VARIANTS = [("Main (RF, 11 features)", "RandomForest", FEATURES, None)]
SENS = plan["sensitivity"]
if SENS["spectral_only"]:
    VARIANTS.append(("RF, 7 spectral features", "RandomForest", ctx.spectral, None))
if SENS["no_gamma"]:
    VARIANTS.append(("RF, 10 features (no gamma)", "RandomForest", [f for f in FEATURES if f != "Rel_Gamma_mean"], None))
if SENS["exclude_gamma_above"]:
    g = SENS["exclude_gamma_above"]
    VARIANTS.append((f"RF, epochs with gamma RP > {g} removed", "RandomForest", FEATURES, g))
    for cohort, d in (("BrainLat", df_bl), ("Greek", df_gr)):
        rest = d[d["Rel_Gamma_mean"] <= g].groupby("subject_id").size()
        print(f"{cohort}: epochs with gamma RP > {g}: {100 * np.mean(d['Rel_Gamma_mean'] > g):.1f}% | "
              f"subjects without any remaining epoch: {d.subject_id.nunique() - len(rest)}")
available = model_factories(ctx)
for m in SENS["classifiers"]:
    if m in available:
        VARIANTS.append((f"{m}, 11 features", m, FEATURES, None))
    else:
        log.alert(f"Sensitivity with {m} skipped (library missing).")

SENS_ROWS = []
for name, model, feats, gamma in VARIANTS:
    print(f"\n— {name}")
    SENS_ROWS.append(variant_summary(name, model, feats, gamma))
TAB_SENS = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("_")} for r in SENS_ROWS])
display(TAB_SENS)
TAB_SENS.to_csv(out / "tables" / "S5_sensitivity.csv", index=False, encoding="utf-8-sig")
for r in SENS_ROWS[1:]:
    log.record("Sensitivity", r["variant"],
               f"RQ1a OR {r['RQ1a OR/SD']} (p {r['RQ1a p']}); RQ1b OR {r['RQ1b OR/SD']} (one-sided p "
               f"{r['RQ1b one-sided p']}); BL->GR {r['RQ3a BL->GR']}; GR->BL {r['RQ3b GR->BL']}",
               "Results; Suppl. Table S5", f"changes the reading of: {r['changes the reading of']}")
write_json(out / "results" / "sensitivity.json", [{"variant": r["variant"], "or": r["_or"]} for r in SENS_ROWS])